# CS-781 Capstone — Phase 1: Legal Language Model Pretraining

Trains a legal language model **from scratch** and generates 128-word continuations for the official
512-word test prompts, implementing **four substantially different architectures** as required:

| `ARCH` value | Architecture family | What makes it structurally different |
|---|---|---|
| `"gpt"` | Decoder-only Transformer | Full causal softmax attention, sinusoidal PE, GELU FFN, post-norm |
| `"prefixlm"` | Encoder-decoder Transformer | Separate bidirectional encoder + causal decoder + cross-attention |
| `"unilm"` | Encoder/hybrid architecture | ONE shared stack, custom mask: bidirectional over the prompt, causal over the continuation (no separate encoder/decoder weights, no cross-attention) |
| `"linear"` | Efficient/lightweight Transformer | Causal **linear attention** (no softmax), RoPE, grouped-query attention, RMSNorm, SwiGLU |

All four are trained **from scratch** on the provided corpus — no pretrained weights are loaded anywhere,
per the competition rules. Every model lands at roughly 75-85M parameters (well inside the 50M-300M limit).

## How to use this notebook on Kaggle

1. Add a Kaggle **Secret** named `HF` holding a Hugging Face access token (Add-ons -> Secrets), so the
   gated dataset can be downloaded. Turn **Internet ON** and pick a **GPU** accelerator (T4x2 or P100).
2. Set `ARCH` in the Config cell to one of `"gpt" | "prefixlm" | "unilm" | "linear"`, then
   **Run All**. Re-run the whole notebook once per architecture (just change `ARCH` and Run All again) —
   each run writes to its own folder and produces its own `submission_<ARCH>.zip`, so the four runs never
   clash.
3. **Given the imminent deadline, train & submit `"gpt"` first** — it is the cheapest and most robust of
   the four. That single submission, resubmitted with 2-3 small, non-identical decoding-parameter tweaks
   (e.g. different `TEMPERATURE`), already satisfies the "minimum 4 submissions" rule for Phase 1. Then
   train the other three architectures over the following days for the architecture-exploration part of
   your report (the report itself isn't due until Nov 8).
4. `MAX_TRAIN_SECONDS` in the Config cell is a hard wall-clock safety cutoff — training always stops and
   checkpoints before it, so a run finishes inside one Kaggle session regardless of `TRAIN_TOKENS`.
   Lower `TRAIN_TOKENS` for a faster first pass; the cell is resumable (re-running the training cell
   picks the checkpoint back up) if your session gets interrupted.
5. The doc recommends registering multiple Kaggle/Colab accounts — handy here since you can train two
   architectures in parallel in separate sessions.

In [ ]:
# Kaggle images already ship torch/transformers/tokenizers/huggingface_hub.
# These installs are cheap no-ops if already present; the metric libraries are OPTIONAL and only used
# by the local sanity-check cell near the end (never for training or for generating predictions).
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("tokenizers", "huggingface_hub")
pip_install("rouge-score", "sacrebleu", "bert-score")  # optional, local sanity-check only
print("Install step done.")

## 1. Hugging Face login

Reads a Kaggle Secret named `HF`. If you're not running on Kaggle (or haven't set the secret), paste a
token directly into `HF_TOKEN` below as a fallback.

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = ""  # <-- paste your Hugging Face token here if Kaggle secrets aren't set up

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("WARNING: no HF token found. Add a Kaggle secret named 'HF' (Add-ons > Secrets), "
          "or paste a token above -- the dataset download will fail without it.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 3. Configuration

The single knob panel for the whole notebook. **Change `ARCH` and re-run the whole notebook** to train
each of the four required architectures.

In [ ]:
import os, json, random, math, time, zipfile

# ============================= CONFIG =============================
ARCH = "gpt"   # one of: "gpt", "prefixlm", "unilm", "linear"
assert ARCH in {"gpt", "prefixlm", "unilm", "linear"}

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Pick which of the four released Phase-1 sources to pretrain on. Acts + Supreme Court are the
# smallest/most curated, which is a reasonable default given the deadline; add
# "Phase_1_Pretraining/Train/train_high_courts.json" and ".../train_tribunals.json" once you have more
# time/compute, and treat the choice itself as one of your data-mixing experiments for the report.
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ---- Tokenizer: trained from scratch on the legal corpus (byte-level BPE, no pretrained vocab) ----
TOKENIZER_VOCAB_SIZE = 16000
TOKENIZER_TRAIN_CHAR_BUDGET = 60_000_000   # characters sampled to TRAIN the tokenizer itself

# ---- Token / data budget for MODEL pretraining ----
TRAIN_TOKENS = 150_000_000   # lower this (e.g. 40_000_000) for a much faster first pass
VAL_TOKENS = 3_000_000
MAX_TRAIN_SECONDS = 6 * 3600   # hard wall-clock safety cutoff per run, independent of TRAIN_TOKENS

# ---- Sequence length ----
# gpt / linear: a single causal block of length BLOCK_SIZE (+1 target token).
# prefixlm / unilm: a context segment + a target segment, mirroring the real 512-word -> 128-word task.
BLOCK_SIZE = 1024
CONTEXT_LEN = 768
TARGET_LEN = 256

# ---- Model dims: chosen so every architecture lands at ~75-85M params (inside the 50M-300M limit) ----
MODEL_DIMS = {
    "gpt":      dict(d_model=768, n_layer=10, n_head=12, d_ff=3072),
    "prefixlm": dict(d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, d_ff=2560),
    "unilm":    dict(d_model=768, n_layer=10, n_head=12, d_ff=3072),
    "linear":   dict(d_model=768, n_layer=10, n_head=12, n_kv_head=4, d_ff=2048),
}

# ---- Training hyperparameters ----
PER_DEVICE_BATCH = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
EVAL_EVERY_STEPS = 500
LOG_EVERY_STEPS = 50

# ---- Generation ----
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 220          # token budget before trimming to 128 words
TEMPERATURE = 0.8
TOP_P = 0.95
GENERATION_BATCH_SIZE = 16

# ---- Paths (architecture-specific, so the four runs never collide) ----
OUTPUT_DIR = f"/kaggle/working/cs781_phase1_{ARCH}"
CKPT_PATH = os.path.join(OUTPUT_DIR, "checkpoint.pt")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "tokenizer.json")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{ARCH}.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Architecture: {ARCH}")
print(f"Output dir:   {OUTPUT_DIR}")

## 4. Download Phase-1 data from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []
for fname in TRAIN_FILES:
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN or None)
    train_file_paths.append(p)
    print("downloaded:", p)

test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN or None)
print("downloaded:", test_file_path)

In [ ]:
def load_records(path):
    """Loads a Phase-1 train file. Defensive to either a list of {"text": ...} records or a
    dict keyed by id -> {"text": ...}."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        return [v["text"] if isinstance(v, dict) else v for v in data.values()]
    return [r["text"] if isinstance(r, dict) else r for r in data]

all_texts = []
for p in train_file_paths:
    recs = load_records(p)
    print(p, "->", len(recs), "documents")
    all_texts.extend(recs)

random.shuffle(all_texts)
print("Total documents pooled:", len(all_texts))

## 5. Train a byte-level BPE tokenizer from scratch on the legal corpus

Per the brief: "You should use the BPE tokenizer trained for the course assignment, or you may train
another tokenizer specifically for this phase." This trains a fresh one (no pretrained GPT-2/other vocab)
directly on the provided legal corpus using the `tokenizers` library's standard BPE trainer.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

if os.path.exists(TOKENIZER_PATH):
    print("Loading cached tokenizer...")
    hf_tok = Tokenizer.from_file(TOKENIZER_PATH)
else:
    print("Training a fresh byte-level BPE tokenizer on the legal corpus...")
    sample_path = os.path.join(OUTPUT_DIR, "tokenizer_train_sample.txt")
    chars_written = 0
    with open(sample_path, "w", encoding="utf-8") as f:
        for text in all_texts:
            if chars_written >= TOKENIZER_TRAIN_CHAR_BUDGET:
                break
            f.write(text.replace("\n", " ") + "\n")
            chars_written += len(text)
    print(f"Tokenizer training sample: {chars_written:,} characters")

    hf_tok = Tokenizer(models.BPE(unk_token="<unk>"))
    hf_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    hf_tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=TOKENIZER_VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"],
    )
    hf_tok.train([sample_path], trainer)
    hf_tok.save(TOKENIZER_PATH)
    print("Tokenizer saved to", TOKENIZER_PATH)

PAD_ID = hf_tok.token_to_id("<pad>")
BOS_ID = hf_tok.token_to_id("<bos>")
EOS_ID = hf_tok.token_to_id("<eos>")
UNK_ID = hf_tok.token_to_id("<unk>")
VOCAB_SIZE = hf_tok.get_vocab_size()
print("Vocab size:", VOCAB_SIZE)
print("special ids -- pad:", PAD_ID, "bos:", BOS_ID, "eos:", EOS_ID, "unk:", UNK_ID)

def encode(text):
    return hf_tok.encode(text).ids

def decode(ids):
    return hf_tok.decode(ids)

_sample = all_texts[0][:300]
print("round-trip check:", decode(encode(_sample))[:150])

## 6. Build training / validation token blocks

A small held-out slice of DOCUMENTS (not just tokens) is reserved for validation, so validation blocks
never share a source document with training blocks.

In [ ]:
def build_token_stream(texts, token_budget, seq_len):
    """Tokenizes documents (each followed by <eos>) and packs them into fixed-length blocks,
    stopping once `token_budget` tokens have been collected."""
    buf = []
    blocks = []
    for text in texts:
        ids = encode(text) + [EOS_ID]
        buf.extend(ids)
        while len(buf) >= seq_len:
            blocks.append(buf[:seq_len])
            buf = buf[seq_len:]
        if len(blocks) * seq_len >= token_budget:
            break
    return blocks

SEQ_LEN = (BLOCK_SIZE + 1) if ARCH in ("gpt", "linear") else (CONTEXT_LEN + TARGET_LEN)

n_val_docs = max(1, int(0.02 * len(all_texts)))
val_texts = all_texts[:n_val_docs]
train_texts = all_texts[n_val_docs:]

print("Building training blocks...")
train_blocks = build_token_stream(train_texts, TRAIN_TOKENS, SEQ_LEN)
print("Building validation blocks...")
val_blocks = build_token_stream(val_texts, VAL_TOKENS, SEQ_LEN)

print(f"Train blocks: {len(train_blocks):,}  (~{len(train_blocks) * SEQ_LEN:,} tokens)")
print(f"Val blocks:   {len(val_blocks):,}  (~{len(val_blocks) * SEQ_LEN:,} tokens)")
assert len(train_blocks) > 0 and len(val_blocks) > 0, "Not enough data -- lower TOKENIZER_TRAIN_CHAR_BUDGET usage or add more TRAIN_FILES."

In [ ]:
from torch.utils.data import Dataset, DataLoader

class BlockDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = blocks
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, idx):
        return torch.tensor(self.blocks[idx], dtype=torch.long)

train_ds = BlockDataset(train_blocks)
val_ds = BlockDataset(val_blocks)

train_loader = DataLoader(train_ds, batch_size=PER_DEVICE_BATCH, shuffle=True, drop_last=True,
                           num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=PER_DEVICE_BATCH, shuffle=False, drop_last=True,
                         num_workers=2, pin_memory=torch.cuda.is_available())

print("train batches/epoch:", len(train_loader), " val batches:", len(val_loader))

## 7. Shared building blocks (used by more than one architecture)

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

def sinusoidal_positional_encoding(max_len, d_model):
    position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    inv_freq = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * inv_freq)
    pe[:, 1::2] = torch.cos(position * inv_freq)
    return pe

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def build_rope_cache(max_len, d_head, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    t = torch.arange(max_len, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)               # (max_len, d_head/2)
    emb = torch.cat([freqs, freqs], dim=-1)          # (max_len, d_head)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    # x: (B, H, T, d_head); cos/sin: (T, d_head)
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    return x * cos + rotate_half(x) * sin

class GPTFeedForward(nn.Module):
    """Plain two-layer GELU FFN, reused by the gpt / prefixlm / unilm architectures."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

print("shared building blocks ready")

## 8. Architecture A -- `gpt`: decoder-only Transformer (full causal softmax attention)

Post-norm GPT-2-style decoder: standard multi-head self-attention with a causal mask, sinusoidal
positional embeddings, GELU feed-forward, tied input/output embeddings.

In [ ]:
class CausalSelfAttentionMHA(nn.Module):
    def __init__(self, d_model, n_head, max_seq_len, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        weights = self.attn_drop(torch.softmax(scores, dim=-1))
        z = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class GPTBlock(nn.Module):
    """Post-norm block (as in GPT-2): LN(x + Attn(x)), then LN(x + FFN(x))."""
    def __init__(self, d_model, n_head, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttentionMHA(d_model, n_head, max_seq_len, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = GPTFeedForward(d_model, d_ff, dropout)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x

class GPTDecoderLM(nn.Module):
    """Architecture 1: vanilla decoder-only Transformer (GPT-2 style)."""
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_ff, max_seq_len, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_head, d_ff, max_seq_len, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) * math.sqrt(self.d_model) + self.pos_emb[:T].unsqueeze(0)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

    def compute_loss(self, batch):
        """batch: (B, BLOCK_SIZE+1). Standard next-token prediction."""
        x, y = batch[:, :-1], batch[:, 1:]
        logits = self(x)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=self.pad_id)

    @torch.no_grad()
    def generate(self, prompt_ids_batch, max_new_tokens, temperature=0.8, top_p=0.95):
        """prompt_ids_batch: list[list[int]]. All prompts in the batch are trimmed to the SAME length
        (keeping each one's last tokens) so no padding/attention-mask handling is needed -- fine here
        since every official test prompt is ~512 words, i.e. already roughly uniform length."""
        self.eval()
        dev = next(self.parameters()).device
        ctx_cap = self.max_seq_len - max_new_tokens
        trimmed = [p[-ctx_cap:] for p in prompt_ids_batch]
        min_len = min(len(p) for p in trimmed)
        x = torch.tensor([p[-min_len:] for p in trimmed], dtype=torch.long, device=dev)
        B = x.size(0)
        generated = [[] for _ in range(B)]
        for _ in range(max_new_tokens):
            logits = self(x[:, -self.max_seq_len:])[:, -1, :]
            logits = logits / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            cutoff = cum > top_p
            cutoff[:, 0] = False
            sorted_probs = sorted_probs.masked_fill(cutoff, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            choice = torch.multinomial(sorted_probs, 1)
            next_tok = sorted_idx.gather(-1, choice).squeeze(-1)
            for i in range(B):
                generated[i].append(int(next_tok[i]))
            x = torch.cat([x, next_tok.unsqueeze(1)], dim=1)
        return generated

print("architecture A (gpt) defined")

## 9. Architecture B -- `prefixlm`: encoder-decoder Transformer

A bidirectional encoder reads the prompt (context); a causal decoder with cross-attention to the
encoder's output generates the continuation (target), trained with teacher forcing.

In [ ]:
class BidirectionalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.scale   # no mask: fully bidirectional, like an encoder
        weights = self.attn_drop(torch.softmax(scores, dim=-1))
        z = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class CrossAttention(nn.Module):
    """Decoder queries attend to encoder outputs (keys/values)."""
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x_dec, enc_out):
        B, T, C = x_dec.shape
        S = enc_out.size(1)
        q = self.q_proj(x_dec).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(enc_out).view(B, S, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_head, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        weights = self.attn_drop(torch.softmax(scores, dim=-1))
        z = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = BidirectionalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = GPTFeedForward(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = CausalSelfAttentionMHA(d_model, n_head, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_head, dropout)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = GPTFeedForward(d_model, d_ff, dropout)
    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ffn(self.ln3(x))
        return x

class PrefixLMEncoderDecoder(nn.Module):
    """Architecture 2: encoder-decoder Transformer."""
    def __init__(self, vocab_size, d_model, n_enc_layer, n_dec_layer, n_head, d_ff,
                 max_ctx_len, max_tgt_len, dropout=0.1, pad_id=0, bos_id=1):
        super().__init__()
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.max_ctx_len = max_ctx_len
        self.max_tgt_len = max_tgt_len
        self.d_model = d_model

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_ctx_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_head, d_ff, dropout) for _ in range(n_enc_layer)])
        self.enc_ln_f = nn.LayerNorm(d_model)

        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_head, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layer)])
        self.dec_ln_f = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def encode(self, ctx_ids):
        T = ctx_ids.size(1)
        x = self.tok_emb(ctx_ids) * math.sqrt(self.d_model) + self.enc_pos[:T].unsqueeze(0)
        x = self.drop(x)
        for block in self.encoder:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_ids, enc_out):
        T = tgt_ids.size(1)
        x = self.tok_emb(tgt_ids) * math.sqrt(self.d_model) + self.dec_pos[:T].unsqueeze(0)
        x = self.drop(x)
        for block in self.decoder:
            x = block(x, enc_out)
        return self.head(self.dec_ln_f(x))

    def compute_loss(self, batch):
        """batch: (B, CONTEXT_LEN + TARGET_LEN). First CONTEXT_LEN tokens = prompt (encoder input);
        remaining TARGET_LEN tokens = continuation (decoder trains with teacher forcing from <bos>)."""
        ctx = batch[:, :self.max_ctx_len]
        tgt = batch[:, self.max_ctx_len:]
        bos_col = torch.full((tgt.size(0), 1), self.bos_id, dtype=torch.long, device=tgt.device)
        dec_in = torch.cat([bos_col, tgt[:, :-1]], dim=1)
        enc_out = self.encode(ctx)
        logits = self.decode(dec_in, enc_out)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=self.pad_id)

    @torch.no_grad()
    def generate(self, prompt_ids_batch, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        dev = next(self.parameters()).device
        trimmed = [p[-self.max_ctx_len:] for p in prompt_ids_batch]
        min_len = min(len(p) for p in trimmed)
        ctx = torch.tensor([p[-min_len:] for p in trimmed], dtype=torch.long, device=dev)
        B = ctx.size(0)
        enc_out = self.encode(ctx)
        dec_in = torch.full((B, 1), self.bos_id, dtype=torch.long, device=dev)
        generated = [[] for _ in range(B)]
        max_new_tokens = min(max_new_tokens, self.max_tgt_len)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_in[:, -self.max_tgt_len:], enc_out)[:, -1, :]
            logits = logits / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            cutoff = cum > top_p
            cutoff[:, 0] = False
            sorted_probs = sorted_probs.masked_fill(cutoff, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            choice = torch.multinomial(sorted_probs, 1)
            next_tok = sorted_idx.gather(-1, choice).squeeze(-1)
            for i in range(B):
                generated[i].append(int(next_tok[i]))
            dec_in = torch.cat([dec_in, next_tok.unsqueeze(1)], dim=1)
        return generated

print("architecture B (prefixlm) defined")

## 10. Architecture C -- `unilm`: encoder/hybrid architecture adapted for LM

ONE shared Transformer stack (no separate encoder/decoder weights, no cross-attention module) processes
`[context, target]` together. A custom attention mask makes the context segment bidirectional (like an
encoder) and the target segment causal (like a decoder) -- the UniLM "sequence-to-sequence LM" recipe.

In [ ]:
class MaskedSelfAttention(nn.Module):
    """Self-attention with an externally supplied boolean attend-mask (True = may attend)."""
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attend_mask):
        """attend_mask: (T, T) bool."""
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~attend_mask[None, None, :, :], float("-inf"))
        weights = self.attn_drop(torch.softmax(scores, dim=-1))
        z = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class UniLMBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MaskedSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = GPTFeedForward(d_model, d_ff, dropout)
    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x

def build_unilm_mask(ctx_len, tgt_len, device):
    """(ctx_len+tgt_len, ctx_len+tgt_len) bool mask: bidirectional within the context block,
    causal (sees all context + earlier target tokens) within the target block."""
    T = ctx_len + tgt_len
    mask = torch.zeros(T, T, dtype=torch.bool, device=device)
    mask[:ctx_len, :ctx_len] = True                      # context sees all of context
    mask[ctx_len:, :ctx_len] = True                       # target sees all of context
    tgt_causal = torch.tril(torch.ones(tgt_len, tgt_len, dtype=torch.bool, device=device))
    mask[ctx_len:, ctx_len:] = tgt_causal                  # target sees earlier target tokens (causal)
    return mask

class UniLMHybridLM(nn.Module):
    """Architecture 3: encoder/hybrid model adapted for LM (UniLM-style)."""
    def __init__(self, vocab_size, d_model, n_layer, n_head, d_ff, max_ctx_len, max_tgt_len,
                 dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.max_ctx_len = max_ctx_len
        self.max_tgt_len = max_tgt_len
        self.max_seq_len = max_ctx_len + max_tgt_len
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(self.max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([UniLMBlock(d_model, n_head, d_ff, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, ids, ctx_len):
        T = ids.size(1)
        tgt_len = T - ctx_len
        mask = build_unilm_mask(ctx_len, tgt_len, ids.device)
        x = self.tok_emb(ids) * math.sqrt(self.d_model) + self.pos_emb[:T].unsqueeze(0)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.ln_f(x))

    def compute_loss(self, batch):
        """batch: (B, CONTEXT_LEN + TARGET_LEN). Loss is computed ONLY on target positions --
        context positions are never a prediction target, exactly like an encoder segment."""
        ctx_len = self.max_ctx_len
        logits = self(batch, ctx_len)
        pred_logits = logits[:, ctx_len - 1:-1, :]
        targets = batch[:, ctx_len:]
        return F.cross_entropy(pred_logits.reshape(-1, pred_logits.size(-1)), targets.reshape(-1),
                                ignore_index=self.pad_id)

    @torch.no_grad()
    def generate(self, prompt_ids_batch, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        dev = next(self.parameters()).device
        trimmed = [p[-self.max_ctx_len:] for p in prompt_ids_batch]
        min_len = min(len(p) for p in trimmed)
        seq = torch.tensor([p[-min_len:] for p in trimmed], dtype=torch.long, device=dev)
        ctx_len = min_len
        B = seq.size(0)
        max_new_tokens = min(max_new_tokens, self.max_tgt_len)
        generated = [[] for _ in range(B)]
        for _ in range(max_new_tokens):
            logits = self(seq, ctx_len)[:, -1, :]
            logits = logits / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            cutoff = cum > top_p
            cutoff[:, 0] = False
            sorted_probs = sorted_probs.masked_fill(cutoff, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            choice = torch.multinomial(sorted_probs, 1)
            next_tok = sorted_idx.gather(-1, choice).squeeze(-1)
            for i in range(B):
                generated[i].append(int(next_tok[i]))
            seq = torch.cat([seq, next_tok.unsqueeze(1)], dim=1)
        return generated

print("architecture C (unilm) defined")

## 11. Architecture D -- `linear`: efficient/lightweight Transformer

Causal **linear attention** (positive kernel feature map instead of softmax), **grouped-query attention**,
**RoPE** instead of an additive positional table, **RMSNorm**, and a **SwiGLU** feed-forward -- a modern
"efficient decoder" recipe, structurally distinct from the other three (no softmax attention, no additive
positional embedding, no LayerNorm).

In [ ]:
class CausalLinearAttentionGQA(nn.Module):
    """Causal linear attention with grouped-query attention and RoPE.
    phi(x) = elu(x) + 1 keeps features non-negative; attention weights are row-sum normalized instead
    of softmax-normalized. This masked-matmul form computes exactly the same quantity as the classic
    O(T) cumulative-sum linear-attention recurrence, just with a simpler/safer implementation."""
    def __init__(self, d_model, n_head, n_kv_head, max_seq_len, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0 and n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.resid_drop = nn.Dropout(dropout)
        cos, sin = build_rope_cache(max_seq_len, self.d_head)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    @staticmethod
    def _feature_map(x):
        return F.elu(x) + 1.0

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        k = k.repeat_interleave(self.n_rep, dim=1)    # expand kv heads to match query heads (GQA)
        v = v.repeat_interleave(self.n_rep, dim=1)

        q = self._feature_map(q)
        k = self._feature_map(k)

        scores = torch.matmul(q, k.transpose(-2, -1))                # (B,H,T,T), NO softmax
        scores = scores.masked_fill(~self.causal_mask[:T, :T], 0.0)
        denom = scores.sum(dim=-1, keepdim=True).clamp(min=1e-6)
        weights = scores / denom
        z = torch.matmul(weights, v)
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(z))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)   # gate
        self.w2 = nn.Linear(d_model, d_ff, bias=False)   # value
        self.w3 = nn.Linear(d_ff, d_model, bias=False)   # out
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class LinearAttnBlock(nn.Module):
    """Pre-norm block with RMSNorm (instead of LayerNorm)."""
    def __init__(self, d_model, n_head, n_kv_head, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalLinearAttentionGQA(d_model, n_head, n_kv_head, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class LinearAttnDecoderLM(nn.Module):
    """Architecture 4: efficient/lightweight decoder."""
    def __init__(self, vocab_size, d_model, n_layer, n_head, n_kv_head, d_ff, max_seq_len, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LinearAttnBlock(d_model, n_head, n_kv_head, d_ff, max_seq_len, dropout) for _ in range(n_layer)
        ])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))     # no additive PE table -- position comes from RoPE inside attention
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm_f(x))

    def compute_loss(self, batch):
        x, y = batch[:, :-1], batch[:, 1:]
        logits = self(x)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=self.pad_id)

    @torch.no_grad()
    def generate(self, prompt_ids_batch, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        dev = next(self.parameters()).device
        ctx_cap = self.max_seq_len - max_new_tokens
        trimmed = [p[-ctx_cap:] for p in prompt_ids_batch]
        min_len = min(len(p) for p in trimmed)
        x = torch.tensor([p[-min_len:] for p in trimmed], dtype=torch.long, device=dev)
        B = x.size(0)
        generated = [[] for _ in range(B)]
        for _ in range(max_new_tokens):
            logits = self(x[:, -self.max_seq_len:])[:, -1, :]
            logits = logits / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            cutoff = cum > top_p
            cutoff[:, 0] = False
            sorted_probs = sorted_probs.masked_fill(cutoff, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            choice = torch.multinomial(sorted_probs, 1)
            next_tok = sorted_idx.gather(-1, choice).squeeze(-1)
            for i in range(B):
                generated[i].append(int(next_tok[i]))
            x = torch.cat([x, next_tok.unsqueeze(1)], dim=1)
        return generated

print("architecture D (linear) defined")

## 12. Build the model selected by `ARCH`

In [ ]:
def build_model():
    dims = MODEL_DIMS[ARCH]
    if ARCH == "gpt":
        return GPTDecoderLM(VOCAB_SIZE, dims["d_model"], dims["n_layer"], dims["n_head"], dims["d_ff"],
                             max_seq_len=BLOCK_SIZE, pad_id=PAD_ID)
    if ARCH == "linear":
        return LinearAttnDecoderLM(VOCAB_SIZE, dims["d_model"], dims["n_layer"], dims["n_head"],
                                    dims["n_kv_head"], dims["d_ff"], max_seq_len=BLOCK_SIZE, pad_id=PAD_ID)
    if ARCH == "prefixlm":
        return PrefixLMEncoderDecoder(VOCAB_SIZE, dims["d_model"], dims["n_enc_layer"], dims["n_dec_layer"],
                                       dims["n_head"], dims["d_ff"], max_ctx_len=CONTEXT_LEN,
                                       max_tgt_len=TARGET_LEN, pad_id=PAD_ID, bos_id=BOS_ID)
    if ARCH == "unilm":
        return UniLMHybridLM(VOCAB_SIZE, dims["d_model"], dims["n_layer"], dims["n_head"], dims["d_ff"],
                              max_ctx_len=CONTEXT_LEN, max_tgt_len=TARGET_LEN, pad_id=PAD_ID)
    raise ValueError(ARCH)

model = build_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Architecture: {ARCH}")
print(f"Model size: {n_params / 1e6:.2f}M parameters (must be between 50M and 300M)")
assert 50e6 <= n_params <= 300e6, "Adjust MODEL_DIMS for this architecture to land in the 50M-300M range."

## 13. Training loop

Generic across all four architectures (each exposes `model.compute_loss(batch)`). AdamW + cosine
schedule with warmup, gradient accumulation, AMP, gradient clipping, periodic validation + checkpointing,
and a hard wall-clock cutoff (`MAX_TRAIN_SECONDS`) so the run always finishes inside one Kaggle session.
Re-running this cell resumes from the last checkpoint.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

total_steps = max(1, len(train_loader) // GRAD_ACCUM_STEPS)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
amp_dtype = torch.float16
autocast_device = "cuda" if torch.cuda.is_available() else "cpu"

start_step = 0
log_history = {"step": [], "train_loss": [], "eval_step": [], "eval_loss": []}

if os.path.exists(CKPT_PATH):
    print("Resuming from checkpoint...")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_step = ckpt["step"]
    log_history = ckpt.get("log_history", log_history)
    print(f"Resumed at step {start_step}")

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for batch in val_loader:
        batch = batch.to(device)
        with torch.autocast(device_type=autocast_device, dtype=amp_dtype, enabled=torch.cuda.is_available()):
            loss = model.compute_loss(batch)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))

def save_checkpoint(step):
    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "step": step, "log_history": log_history,
    }, CKPT_PATH)

print(f"Planned optimizer steps this run: {total_steps:,} (effective batch = {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS})")
print(f"Hard time budget: {MAX_TRAIN_SECONDS/3600:.1f}h")

model.train()
step = start_step
t_start = time.time()
micro_step = 0
optimizer.zero_grad()
stop = (step >= total_steps)

while step < total_steps and not stop:
    for batch in train_loader:
        batch = batch.to(device)
        with torch.autocast(device_type=autocast_device, dtype=amp_dtype, enabled=torch.cuda.is_available()):
            loss = model.compute_loss(batch) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        micro_step += 1

        if micro_step % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

            if step % LOG_EVERY_STEPS == 0:
                elapsed = time.time() - t_start
                log_history["step"].append(step)
                log_history["train_loss"].append(loss.item() * GRAD_ACCUM_STEPS)
                print(f"step {step:>6}/{total_steps} | loss {loss.item() * GRAD_ACCUM_STEPS:.4f} "
                      f"| lr {scheduler.get_last_lr()[0]:.2e} | {elapsed/60:.1f} min elapsed")

            if step % EVAL_EVERY_STEPS == 0:
                eval_loss = evaluate()
                log_history["eval_step"].append(step)
                log_history["eval_loss"].append(eval_loss)
                print(f"  eval @ step {step}: loss {eval_loss:.4f} | perplexity {math.exp(min(eval_loss, 20)):.2f}")
                save_checkpoint(step)

            if time.time() - t_start > MAX_TRAIN_SECONDS:
                print(f"Hit the {MAX_TRAIN_SECONDS/3600:.1f}h time budget -- stopping training early.")
                stop = True
            if step >= total_steps:
                stop = True
        if stop:
            break

save_checkpoint(step)
final_eval_loss = evaluate()
print(f"Final validation loss: {final_eval_loss:.4f} | perplexity: {math.exp(min(final_eval_loss, 20)):.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(log_history["step"], log_history["train_loss"], label="train loss")
if log_history["eval_step"]:
    plt.plot(log_history["eval_step"], log_history["eval_loss"], label="val loss", marker="o")
plt.xlabel("optimizer step")
plt.ylabel("loss")
plt.title(f"Phase 1 [{ARCH}] training / validation loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=150)
plt.show()
print("saved:", LOSS_PLOT_PATH)

## 14. Generate predictions for the official test prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("test examples:", len(test_data))

model.eval()
print("Tokenizing test prompts...")
test_items = list(test_data.items())
test_encoded = {k: encode(v["text"]) for k, v in test_items}
# Sort by token length so each generation batch trims as little as possible (model.generate aligns
# every prompt in a batch to the shortest one in that batch).
test_items_sorted = sorted(test_items, key=lambda kv: len(test_encoded[kv[0]]))

predictions = {}
for start in range(0, len(test_items_sorted), GENERATION_BATCH_SIZE):
    batch_items = test_items_sorted[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [k for k, _ in batch_items]
    batch_prompt_ids = [test_encoded[k] for k in batch_ids]

    gen_token_lists = model.generate(batch_prompt_ids, MAX_GENERATION_TOKENS,
                                      temperature=TEMPERATURE, top_p=TOP_P)
    for ex_id, gen_ids in zip(batch_ids, gen_token_lists):
        text = decode(gen_ids)
        words = text.split()[:TARGET_GENERATION_WORDS]
        predictions[ex_id] = {"text": " ".join(words)}

    done = start + len(batch_items)
    print(f"generated {done}/{len(test_items_sorted)} ({100*done/len(test_items_sorted):.1f}%)")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("saved:", PREDICTION_PATH)

word_counts = [len(v["text"].split()) for v in predictions.values()]
print(f"generated word counts: min={min(word_counts)} max={max(word_counts)} mean={sum(word_counts)/len(word_counts):.1f}")
print("(if counts are persistently well under 128, raise MAX_GENERATION_TOKENS and regenerate)")

## 15. Package and validate the Codabench submission

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission ready:", SUBMISSION_ZIP)

with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    assert names == ["prediction.json"], f"zip must contain only prediction.json at the root, got {names}"
print("zip structure OK: prediction.json at archive root, no extra folders.")

with open(PREDICTION_PATH, encoding="utf-8") as f:
    pred_check = json.load(f)
test_ids = set(test_data.keys())
pred_ids = set(pred_check.keys())
missing = test_ids - pred_ids
extra = pred_ids - test_ids
print(f"missing IDs: {len(missing)} | extra/unknown IDs: {len(extra)}")
assert not missing and not extra, "prediction.json IDs must exactly match the official test IDs."
print("ID coverage OK -- ready to upload", SUBMISSION_ZIP, "to Codabench.")

## 16. (Optional) Local sanity-check metrics

This is **not** the official Codabench score -- the real reference continuations are private. This cell
builds its own prompt/continuation pairs from held-out training documents (first ~512 words as the
prompt, the next ~128 words as the "reference") purely as a local sanity check before spending a
Codabench submission, per Section 16 of the brief.

In [ ]:
def make_local_eval_examples(texts, n=20, prompt_words=512, cont_words=128):
    examples = []
    for text in texts:
        words = text.split()
        if len(words) < prompt_words + cont_words:
            continue
        prompt = " ".join(words[:prompt_words])
        reference = " ".join(words[prompt_words:prompt_words + cont_words])
        examples.append((prompt, reference))
        if len(examples) >= n:
            break
    return examples

local_examples = make_local_eval_examples(val_texts, n=20)
print(f"local eval examples: {len(local_examples)}")

if local_examples:
    prompt_ids_batch = [encode(p) for p, _ in local_examples]
    gen_ids = model.generate(prompt_ids_batch, MAX_GENERATION_TOKENS, temperature=TEMPERATURE, top_p=TOP_P)
    local_preds = [" ".join(decode(g).split()[:TARGET_GENERATION_WORDS]) for g in gen_ids]
    local_refs = [r for _, r in local_examples]

    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        rouge_l = sum(scorer.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(local_preds, local_refs)) / len(local_preds)
        print(f"local ROUGE-L F1 (proxy, NOT the official score): {rouge_l:.4f}")
    except Exception as e:
        print("rouge-score unavailable:", e)

    try:
        import sacrebleu
        bleu = sacrebleu.corpus_bleu(local_preds, [local_refs])
        print(f"local BLEU-4 (proxy): {bleu.score:.2f}")
    except Exception as e:
        print("sacrebleu unavailable:", e)

    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(local_preds, local_refs, lang="en", verbose=False)
        print(f"local BERTScore F1 (proxy): {F1.mean().item():.4f}")
    except Exception as e:
        print("bert-score unavailable (first run downloads a scoring model):", e)

    print("\nNote: these numbers use a locally-built prompt/continuation split from the TRAINING data,")
    print("NOT the official private test references -- treat them only as a rough sanity check.")
else:
    print("No local eval examples could be built (documents too short) -- skip this cell.")

## Before you submit to Codabench

- `submission_<ARCH>.zip` contains **only** `prediction.json` at the archive root (checked in cell 15).
- Every official test ID is present, no extra IDs (checked in cell 15).
- `text` fields contain only the generated continuation, never the original prompt.
- Log this run's config + metrics somewhere for the report: architecture, data mix, token budget,
  context length, final train/val loss, perplexity, generation settings, local proxy metrics.

## Remaining work for the full Phase-1 requirement

- Repeat this notebook for all four `ARCH` values (`gpt`, `prefixlm`, `unilm`, `linear`) and submit each
  -- this alone gives four distinct, non-identical submissions.
- The brief requires 4+ submissions minimum (not capped at the four architectures): once all four
  architectures have a baseline run, additional submissions from checkpoint/decoding variations of your
  best architecture also count, as long as they are not identical repeats.
- Write up the Expected Analysis bullets (Sections 7.4 / 7.9 of the brief) using the loss curves,
  perplexities, and local proxy metrics captured above -- which data mix, context length, architecture,
  and decoding strategy worked best, and why.